# Sentinel-1 + Sentinel-2 RF

## Setup

In [ ]:
#configures, features, and model settings.
RUN_INSTALLS = False
if RUN_INSTALLS:
    %pip -q install earthengine-api geemap geopandas scikit-learn seaborn

from pathlib import Path
import json
import re
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score

RANDOM_SEED = 42
TEST_SIZE = 0.3
REFERENCE_NOTE = 'CROME treated as a reference crop-map product rather than direct ground truth'
VALIDATION_NOTE = 'random train-test split used for the paired sensor comparison'

def locate_stage(stage_name, start=None):
    start = Path.cwd() if start is None else Path(start)
    for root in [start, *start.parents]:
        for parent in (root, root / 'code'):
            candidate = parent / stage_name
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'Could not locate {stage_name}. Run from the dissertation project or stage folder.')

WORK_DIR = locate_stage('2_sentinel1_sentinel2_model')
BASELINE_DIR = WORK_DIR.parent / '1_sentinel_2_baseline'
DATA_PROCESSED = WORK_DIR / 'data' / 'processed'
OUTPUTS = WORK_DIR / 'outputs'
TABLES = OUTPUTS / 'tables'
FIGURES = OUTPUTS / 'figures'
LOGS = OUTPUTS / 'logs'
for folder in [DATA_PROCESSED, TABLES, FIGURES, LOGS]:
    folder.mkdir(parents=True, exist_ok=True)

CLASS_LABELS = [
    'Winter wheat',
    'Winter barley',
    'Spring barley',
    'Beet (sugar beet / fodder beet)',
    'Maize',
    'Oilseed rape',
    'Potatoes',
    'Pulses / field beans and peas',
]
SEASONAL_WINDOWS = {
    'winter_establishment': ('2021-10-20', '2022-02-28'),
    'spring_growth': ('2022-03-01', '2022-05-31'),
    'summer_peak': ('2022-06-01', '2022-08-31'),
    'late_season': ('2022-09-01', '2022-09-30'),
}
S2_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
S2_INDICES = ['NDVI', 'NDRE', 'LSWI', 'EVI']
S1_BASE_FEATURES = ['VV', 'VH', 'VV_minus_VH', 'VV_div_VH']
S2_FEATURE_BANDS = [f'{season}_{band}' for season in SEASONAL_WINDOWS for band in (S2_BANDS + S2_INDICES)]
S1_FEATURE_BANDS = [f'{season}_{band}' for season in SEASONAL_WINDOWS for band in S1_BASE_FEATURES]

S2_SAMPLE_CSV = DATA_PROCESSED / 's2_2022_east_anglia_crome_8class_baseline.csv'
S1S2_SAMPLE_CSV = DATA_PROCESSED / 's1s2_2022_east_anglia_crome_8class_features.csv'
RUN_MANIFEST = LOGS / 's1s2_run_manifest.json'

pd.DataFrame({'band_name': S1_FEATURE_BANDS}).to_csv(TABLES / 's1_feature_band_list.csv', index=False)
pd.DataFrame({'band_name': S2_FEATURE_BANDS}).to_csv(TABLES / 's2_feature_band_list_from_notebook.csv', index=False)
print('Work dir:', WORK_DIR)
print('S1 feature count:', len(S1_FEATURE_BANDS))
print('S2 feature count:', len(S2_FEATURE_BANDS))


## Preserve baseline outputs

In [ ]:
#Copy the baseline outputs.
def copy_if_exists(src, dst):
    src = Path(src)
    dst = Path(dst)
    if src.exists() and not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        return True
    return False
if BASELINE_DIR.exists():
    baseline_copies = [
        ('outputs/tables/s2_only_confusion_matrix.csv', TABLES / 's2_only_confusion_matrix.csv'),
        ('outputs/tables/s2_only_class_metrics.csv', TABLES / 's2_only_class_metrics.csv'),
        ('outputs/tables/s2_only_overall_metrics.csv', TABLES / 's2_only_overall_metrics.csv'),
        ('outputs/tables/s2_only_feature_importance.csv', TABLES / 's2_only_feature_importance.csv'),
        ('outputs/tables/s2_feature_band_list.csv', TABLES / 's2_feature_band_list.csv'),
        ('outputs/figures/s2_only_confusion_matrix.png', FIGURES / 's2_only_confusion_matrix.png'),
        ('processed_data/s2_2022_east_anglia_crome_8class_baseline.csv', S2_SAMPLE_CSV),
    ]
    copied = [str(dst) for rel, dst in baseline_copies if copy_if_exists(BASELINE_DIR / rel, dst)]
    print('Copied baseline files:', len(copied))
else:
    print('Baseline directory not found. Continue if required files are already in this folder.')
print('Baseline preservation/check cell complete.')

## Export Earth Engine features

In [ ]:
#Build and export combined S1 and S2 features.
RUN_GEE_EXPORT = False
EE_PROJECT_ID = None  
S1S2_EXPORT_DRIVE_FOLDER = 'dissertation_baseline_exports'
S1S2_EXPORT_DESCRIPTION = 's1s2_2022_east_anglia_crome_8class_features'
SAMPLE_POINTS_GPKG = BASELINE_DIR / 'processed_data' / 'crome_2022_east_anglia_8class_points_baseline.gpkg'
SAMPLE_POINTS_LAYER = 'points_baseline'
if RUN_GEE_EXPORT:
    import ee
    import geemap
    import geopandas as gpd
    def initialize_ee(project_id=None):
        try:
            if project_id:
                ee.Initialize(project=project_id)
            else:
                ee.Initialize()
        except Exception:
            ee.Authenticate()
            if project_id:
                ee.Initialize(project=project_id)
            else:
                ee.Initialize()
    def mask_s2_sr_harmonized(image):
        scl = image.select('SCL')
        clear = scl.neq(0).And(scl.neq(1)).And(scl.neq(3)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
        return image.updateMask(clear).divide(10000).copyProperties(image, ['system:time_start'])
    def add_s2_indices(image):
        ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
        ndre = image.normalizedDifference(['B8A', 'B5']).rename('NDRE')
        lswi = image.normalizedDifference(['B8', 'B11']).rename('LSWI')
        evi = image.expression(
            '2.5 * ((nir - red) / (nir + 6 * red - 7.5 * blue + 1))',
            {'nir': image.select('B8'), 'red': image.select('B4'), 'blue': image.select('B2')},
        ).rename('EVI')
        return image.addBands([ndvi, ndre, lswi, evi])
    def seasonal_s2_composite(aoi, start, end, season_name):
        collection = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filterDate(start, end)
            .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 70))
            .select(S2_BANDS + ['SCL'])
            .map(mask_s2_sr_harmonized)
            .map(add_s2_indices)
        )
        predictors = S2_BANDS + S2_INDICES
        return collection.select(predictors).median().rename([f'{season_name}_{band}' for band in predictors])
    def add_s1_derived(image):
        vv = image.select('VV')
        vh = image.select('VH')
        vv_linear = ee.Image.constant(10).pow(vv.divide(10))
        vh_linear = ee.Image.constant(10).pow(vh.divide(10))
        ratio = vv_linear.divide(vh_linear.max(ee.Image.constant(1e-6))).rename('VV_div_VH')
        diff = vv.subtract(vh).rename('VV_minus_VH')
        return image.addBands([diff, ratio])
    def seasonal_s1_composite(aoi, start, end, season_name):
        collection = (
            ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(aoi)
            .filterDate(start, end)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
            .select(['VV', 'VH'])
            .map(add_s1_derived)
        )
        return collection.select(S1_BASE_FEATURES).median().rename([f'{season_name}_{band}' for band in S1_BASE_FEATURES])
    def build_s2_feature_stack(aoi):
        return ee.Image.cat([seasonal_s2_composite(aoi, start, end, season) for season, (start, end) in SEASONAL_WINDOWS.items()])
    def build_s1_feature_stack(aoi):
        return ee.Image.cat([seasonal_s1_composite(aoi, start, end, season) for season, (start, end) in SEASONAL_WINDOWS.items()])
    initialize_ee(EE_PROJECT_ID)
    if not SAMPLE_POINTS_GPKG.exists():
        raise FileNotFoundError(f'Missing sample points GeoPackage: {SAMPLE_POINTS_GPKG}')
    samples = gpd.read_file(SAMPLE_POINTS_GPKG, layer=SAMPLE_POINTS_LAYER).to_crs('EPSG:4326')
    keep_cols = [col for col in ['sample_uid', 'class_id', 'analysis_class'] if col in samples.columns]
    samples = samples[keep_cols + ['geometry']]
    sample_fc = geemap.geopandas_to_ee(samples, geodesic=False)
    aoi = sample_fc.geometry().bounds().buffer(20000)
    feature_stack = build_s2_feature_stack(aoi).addBands(build_s1_feature_stack(aoi))
    sampled = feature_stack.sampleRegions(
        collection=sample_fc,
        properties=['sample_uid', 'class_id', 'analysis_class'],
        scale=10,
        geometries=False,
        tileScale=4,
    )
    task = ee.batch.Export.table.toDrive(
        collection=sampled,
        description=S1S2_EXPORT_DESCRIPTION,
        folder=S1S2_EXPORT_DRIVE_FOLDER,
        fileFormat='CSV',
    )
    task.start()
    print('Started Earth Engine task:', S1S2_EXPORT_DESCRIPTION)
else:
    print('RUN_GEE_EXPORT is False.')

## Train the combined RF model

In [ ]:
#Train and evaluate the combined RF model.
def clean_model_frame(df, feature_cols):
    required = ['analysis_class', 'class_id']
    missing_required = [col for col in required if col not in df.columns]
    if missing_required:
        raise ValueError(f'Missing required columns: {missing_required}')
    out = df.copy()
    out = out[out['analysis_class'].isin(CLASS_LABELS)].copy()
    for col in feature_cols:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out = out.dropna(subset=feature_cols + ['analysis_class'])
    return out
def run_rf_model(model_df, feature_cols, prefix):
    X = model_df[feature_cols]
    y = model_df['analysis_class']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
    )
    rf = RandomForestClassifier(
        n_estimators=500,
        class_weight='balanced_subsample',
        max_features='sqrt',
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)
    cm = confusion_matrix(y_test, pred, labels=CLASS_LABELS)
    cm_df = pd.DataFrame(cm, index=CLASS_LABELS, columns=CLASS_LABELS)
    cm_df.to_csv(TABLES / f'{prefix}_confusion_matrix.csv')
    report = classification_report(y_test, pred, labels=CLASS_LABELS, output_dict=True, zero_division=0)
    class_metrics = pd.DataFrame(report).T.loc[CLASS_LABELS, ['precision', 'recall', 'f1-score', 'support']]
    class_metrics = class_metrics.rename(columns={'precision': 'user_accuracy', 'recall': 'producer_accuracy'})
    class_metrics.to_csv(TABLES / f'{prefix}_class_metrics.csv')
    overall = {
        'overall_agreement': accuracy_score(y_test, pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, pred),
        'macro_f1': f1_score(y_test, pred, labels=CLASS_LABELS, average='macro'),
        'weighted_f1': f1_score(y_test, pred, labels=CLASS_LABELS, average='weighted'),
        'validation_note': 'Random split baseline; interpret with spatial leakage risk.',
    }
    pd.DataFrame([overall]).to_csv(TABLES / f'{prefix}_overall_metrics.csv', index=False)
    importance = pd.DataFrame({'feature': feature_cols, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
    importance.to_csv(TABLES / f'{prefix}_feature_importance.csv', index=False)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.xlabel('Predicted class')
    plt.ylabel('CROME reference label')
    plt.title('Sentinel-1 + Sentinel-2 Random Forest agreement with CROME')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURES / f'{prefix}_confusion_matrix.png', dpi=300)
    plt.show()
    return rf, class_metrics, pd.DataFrame([overall]), importance
if S1S2_SAMPLE_CSV.exists():
    s1s2 = pd.read_csv(S1S2_SAMPLE_CSV)
    s2_cols = [col for col in S2_FEATURE_BANDS if col in s1s2.columns]
    s1_cols = [col for col in S1_FEATURE_BANDS if col in s1s2.columns]
    feature_cols = s2_cols + s1_cols
    if not s1_cols:
        raise ValueError('No Sentinel-1 feature columns found in combined feature CSV.')
    model_df = clean_model_frame(s1s2, feature_cols)
    rf, s1s2_class_metrics, s1s2_overall, s1s2_importance = run_rf_model(model_df, feature_cols, 's1s2')
    run_manifest = {
        'reference_product': 'CROME 2022',
        'input_feature_table': str(S1S2_SAMPLE_CSV),
        'rows_used': int(len(model_df)),
        'feature_counts': {
            'sentinel_2': int(len(s2_cols)),
            'sentinel_1': int(len(s1_cols)),
            'total': int(len(feature_cols)),
        },
        'seasonal_windows': SEASONAL_WINDOWS,
        'earth_engine_sources': {
            'sentinel_2': 'COPERNICUS/S2_SR_HARMONIZED',
            'sentinel_1': 'COPERNICUS/S1_GRD',
            'sampling_scale_m': 10,
            'tile_scale': 4,
        },
        'split': {
            'test_size': TEST_SIZE,
            'stratified_by': 'analysis_class',
            'random_seed': RANDOM_SEED,
        },
        'random_forest': {
            'n_estimators': 500,
            'max_features': 'sqrt',
            'class_weight': 'balanced_subsample',
            'random_seed': RANDOM_SEED,
        },
        'reference_note': REFERENCE_NOTE,
    }
    RUN_MANIFEST.write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')
    display(s1s2_overall)
    display(s1s2_class_metrics)
else:
    print(f'Missing combined feature table: {S1S2_SAMPLE_CSV}')
    print('Run the Earth Engine export, download/copy the CSV to this path, then rerun this cell.')


## Compare S2 and S1+S2

In [ ]:
#Compare S2 only and S1+S2 results.
s2_class_path = TABLES / 's2_only_class_metrics.csv'
s1s2_class_path = TABLES / 's1s2_class_metrics.csv'
s2_overall_path = TABLES / 's2_only_overall_metrics.csv'
s1s2_overall_path = TABLES / 's1s2_overall_metrics.csv'
if s2_class_path.exists() and s1s2_class_path.exists():
    s2_cls = pd.read_csv(s2_class_path, index_col=0)
    s1s2_cls = pd.read_csv(s1s2_class_path, index_col=0)
    rows = []
    for cls in CLASS_LABELS:
        rows.append({
            'class': cls,
            's2_user_accuracy': s2_cls.loc[cls, 'user_accuracy'],
            's1s2_user_accuracy': s1s2_cls.loc[cls, 'user_accuracy'],
            'delta_user_accuracy': s1s2_cls.loc[cls, 'user_accuracy'] - s2_cls.loc[cls, 'user_accuracy'],
            's2_producer_accuracy': s2_cls.loc[cls, 'producer_accuracy'],
            's1s2_producer_accuracy': s1s2_cls.loc[cls, 'producer_accuracy'],
            'delta_producer_accuracy': s1s2_cls.loc[cls, 'producer_accuracy'] - s2_cls.loc[cls, 'producer_accuracy'],
            's2_f1': s2_cls.loc[cls, 'f1-score'],
            's1s2_f1': s1s2_cls.loc[cls, 'f1-score'],
            'delta_f1': s1s2_cls.loc[cls, 'f1-score'] - s2_cls.loc[cls, 'f1-score'],
            's2_support': s2_cls.loc[cls, 'support'],
            's1s2_support': s1s2_cls.loc[cls, 'support'],
        })
    comparison = pd.DataFrame(rows)
    comparison.to_csv(TABLES / 'model_comparison_s2_vs_s1s2.csv', index=False)
    s2_overall = pd.read_csv(s2_overall_path).iloc[0]
    s1s2_overall = pd.read_csv(s1s2_overall_path).iloc[0]
    overall_rows = []
    for metric in ['overall_agreement', 'balanced_accuracy', 'macro_f1', 'weighted_f1']:
        delta = s1s2_overall[metric] - s2_overall[metric]
        note = 'higher agreement with CROME labels' if delta > 0 else 'lower agreement with CROME labels' if delta < 0 else 'no change'
        overall_rows.append({'metric': metric, 's2_only': s2_overall[metric], 's1s2': s1s2_overall[metric], 'delta': delta, 'interpretation_note': note})
    overall_comparison = pd.DataFrame(overall_rows)
    overall_comparison.to_csv(TABLES / 'overall_model_comparison_s2_vs_s1s2.csv', index=False)
    plot_df = comparison[['class', 's2_f1', 's1s2_f1']].melt(id_vars='class', var_name='model', value_name='f1')
    plot_df['model'] = plot_df['model'].map({'s2_f1': 'S2-only', 's1s2_f1': 'S1+S2'})
    plt.figure(figsize=(11, 5.5))
    sns.barplot(data=plot_df, x='class', y='f1', hue='model')
    plt.ylim(0, 1)
    plt.xlabel('Crop class')
    plt.ylabel('F1-score')
    plt.title('Class-level F1 agreement with CROME labels')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES / 'f1_comparison_s2_vs_s1s2.png', dpi=300)
    plt.show()
    weak_classes = comparison[comparison['class'].isin(['Pulses / field beans and peas', 'Maize', 'Potatoes'])]
    display(overall_comparison)
    display(weak_classes[['class', 's2_f1', 's1s2_f1', 'delta_f1']])
else:
    print('Comparison skipped because S1+S2 metrics are not available yet.')

## Holdout county validation

In [ ]:
#Evaluate holdout counties.
def infer_county(sample_uid):
    text = str(sample_uid).lower()
    if 'suffolk' in text:
        return 'Suffolk'
    if 'norfolk' in text:
        return 'Norfolk'
    if 'cambridgeshire' in text:
        return 'Cambridgeshire'
    return np.nan
def county_holdout_metrics(df, feature_cols, model_name):
    records = []
    class_records = []
    for county in ['Suffolk', 'Norfolk', 'Cambridgeshire']:
        train = df[df['county'] != county]
        test = df[df['county'] == county]
        if train.empty or test.empty or train['analysis_class'].nunique() < 2 or test['analysis_class'].nunique() < 2:
            continue
        rf = RandomForestClassifier(n_estimators=500, class_weight='balanced_subsample', max_features='sqrt', random_state=RANDOM_SEED, n_jobs=-1)
        rf.fit(train[feature_cols], train['analysis_class'])
        pred = rf.predict(test[feature_cols])
        records.append({
            'model': model_name,
            'held_out_county': county,
            'train_rows': len(train),
            'test_rows': len(test),
            'overall_agreement': accuracy_score(test['analysis_class'], pred),
            'balanced_accuracy': balanced_accuracy_score(test['analysis_class'], pred),
            'macro_f1': f1_score(test['analysis_class'], pred, labels=CLASS_LABELS, average='macro'),
            'reference_note': REFERENCE_NOTE,
        })
        report = classification_report(test['analysis_class'], pred, labels=CLASS_LABELS, output_dict=True, zero_division=0)
        cls = pd.DataFrame(report).T.loc[CLASS_LABELS, ['precision', 'recall', 'f1-score', 'support']].reset_index().rename(columns={'index': 'class', 'precision': 'user_accuracy', 'recall': 'producer_accuracy'})
        cls.insert(0, 'held_out_county', county)
        cls.insert(0, 'model', model_name)
        class_records.append(cls)
    return pd.DataFrame(records), pd.concat(class_records, ignore_index=True) if class_records else pd.DataFrame()
if S1S2_SAMPLE_CSV.exists():
    df = pd.read_csv(S1S2_SAMPLE_CSV)
    s2_cols = [col for col in S2_FEATURE_BANDS if col in df.columns]
    s1_cols = [col for col in S1_FEATURE_BANDS if col in df.columns]
    df['county'] = df.get('sample_uid', pd.Series(index=df.index, dtype='object')).map(infer_county)
    if df['county'].notna().any():
        s2_df = clean_model_frame(df.dropna(subset=['county']), s2_cols)
        s1s2_df = clean_model_frame(df.dropna(subset=['county']), s2_cols + s1_cols)
        metrics_frames = []
        class_frames = []
        if s2_cols:
            m, c = county_holdout_metrics(s2_df, s2_cols, 'S2-only matched rows')
            metrics_frames.append(m); class_frames.append(c)
        if s1_cols:
            m, c = county_holdout_metrics(s1s2_df, s2_cols + s1_cols, 'S1+S2')
            metrics_frames.append(m); class_frames.append(c)
        spatial_metrics = pd.concat(metrics_frames, ignore_index=True)
        spatial_class_metrics = pd.concat(class_frames, ignore_index=True)
        spatial_metrics.to_csv(TABLES / 'spatial_holdout_county_metrics.csv', index=False)
        spatial_class_metrics.to_csv(TABLES / 'spatial_holdout_county_class_metrics.csv', index=False)
        display(spatial_metrics)
    else:
        note = 'County could not be inferred from sample_uid. Spatial validation deferred; discuss spatial leakage risk for random-split metrics.'
        print(note)
else:
    note = 'S1+S2 feature table is not available yet. Spatial validation deferred until the combined feature table is exported and copied locally. Random-split leakage risk must be discussed.'
    print(note)

## Output check

In [ ]:
#Check expected outputs.
required = [
    TABLES / 's1_feature_band_list.csv',
    S1S2_SAMPLE_CSV,
    TABLES / 's1s2_confusion_matrix.csv',
    TABLES / 's1s2_class_metrics.csv',
    TABLES / 's1s2_overall_metrics.csv',
    TABLES / 's1s2_feature_importance.csv',
    FIGURES / 's1s2_confusion_matrix.png',
    TABLES / 'model_comparison_s2_vs_s1s2.csv',
    TABLES / 'overall_model_comparison_s2_vs_s1s2.csv',
    FIGURES / 'f1_comparison_s2_vs_s1s2.png',
    RUN_MANIFEST,
]
check = pd.DataFrame({'path': [str(p.relative_to(WORK_DIR)) if p.is_absolute() and WORK_DIR in p.parents else str(p) for p in required], 'exists': [p.exists() for p in required]})
display(check)
